##  Carga de las fuentes de información

In [1]:
# Instala las librerías a usar
!pip install -q gdown
!pip install -q chromadb
!pip install -q sentence-transformers
!pip install -q langchain-text-splitters
!pip install -q "grafitodb[viz]" matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 95.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 140.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 106.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the

In [33]:
import gdown
import zipfile
import os
import pandas as pd
import json
from pathlib import Path
import chromadb
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import CharacterTextSplitter
from grafito import GrafitoDatabase

###  Descarga del dataset desde Google Drive

In [3]:
file_id = "1LY9FWZzuB-KSrdWkHAMDtNk2LemWLxls" #ID del archivo zip en google
output = "fuentes_de_informacion.zip"

gdown.download(id=file_id, output=output, quiet=False)

Downloading...
From: https://drive.google.com/uc?id=1LY9FWZzuB-KSrdWkHAMDtNk2LemWLxls
To: /content/fuentes_de_informacion.zip
100%|██████████| 3.81M/3.81M [00:00<00:00, 139MB/s]


'fuentes_de_informacion.zip'

### Descompresión del archivo ZIP

In [4]:
# Carpeta donde vamos a descomprimir todo
extract_path = "/content/fuentes_de_informacion"

# Crea la carpeta si no existe
os.makedirs(extract_path, exist_ok=True)

# Abro el ZIP y extraigo todo adentro de extract_path
with zipfile.ZipFile(output, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("ZIP descomprimido")

ZIP descomprimido


In [6]:
# Ruta base real del dataset.
# Quedó una carpeta fuentes_de_informacion adentro de otra por cómo estaba armado el ZIP.
base_path = Path("/content/fuentes_de_informacion/fuentes_de_informacion")

# Rutas a las carpetas de textos
resenas_path = base_path / "resenas_usuarios"
manuales_path = base_path / "manuales_productos"

# Rutas a archivos principales
productos_csv_path = base_path / "productos.csv"
productos_xlsx_path = base_path / "productos.xlsx"
inventario_path = base_path / "inventario_sucursales.csv"
ventas_path = base_path / "ventas_historicas.csv"
devoluciones_path = base_path / "devoluciones.csv"
tickets_path = base_path / "tickets_soporte.csv"
vendedores_path = base_path / "vendedores.csv"
faqs_path = base_path / "faqs.json"

# Verificamos rápido que las rutas existan
print("Base:", base_path.exists())
print("Reseñas:", resenas_path.exists())
print("Manuales:", manuales_path.exists())
print("Productos CSV:", productos_csv_path.exists())
print("FAQs:", faqs_path.exists())

Base: True
Reseñas: True
Manuales: True
Productos CSV: True
FAQs: True


##  Diseño de las fuentes de datos

El sistema utilizará tres fuentes de conocimiento con propósitos distintos:

- Base vectorial: para recuperar información textual no estructurada mediante similitud semántica.
- Base tabular: para responder consultas que requieren filtros, comparaciones, rangos o agregaciones.
- Base de grafos: para representar relaciones entre productos, categorías, subcategorías y marcas.

Esta separación va a permitir elegir la fuente más adecuada según la intención de la consulta del usuario.

In [8]:
# Resumen de diseño de las fuentes que vamos a usar en el sistema

disenio_fuentes = pd.DataFrame([
    {
        "fuente": "Base vectorial",
        "motor": "ChromaDB",
        "informacion": "Manuales, FAQs, reseñas y descripciones textuales de tickets de soporte",
        "uso": "Preguntas sobre uso de productos, opiniones, problemas frecuentes y respuestas en lenguaje natural"
    },
    {
        "fuente": "Base tabular",
        "motor": "Pandas",
        "informacion": "Productos, inventario, ventas, devoluciones, tickets y vendedores",
        "uso": "Consultas con filtros, precios, stock, categorías, ventas, devoluciones y métricas"
    },
    {
        "fuente": "Base de grafos",
        "motor": "GrafitoDB",
        "informacion": "Relaciones entre productos, categorías, subcategorías y marcas",
        "uso": "Consultas sobre relaciones, productos conectados y navegación por categorías"
    }
])

display(disenio_fuentes)

,fuente,motor,informacion,uso
0,Base vectorial,ChromaDB,"Manuales, FAQs, reseñas y descripciones textua...","Preguntas sobre uso de productos, opiniones, p..."
1,Base tabular,Pandas,"Productos, inventario, ventas, devoluciones, t...","Consultas con filtros, precios, stock, categor..."
2,Base de grafos,GrafitoDB,"Relaciones entre productos, categorías, subcat...","Consultas sobre relaciones, productos conectad..."


##  Preparación de documentos para la base vectorial

In [9]:
documentos_vectoriales = []

# 1) FAQs
# Las FAQs sirven para preguntas frecuentes directas de usuarios.
with open(faqs_path, "r", encoding="utf-8") as f:
    faqs_data = json.load(f)

for i, faq in enumerate(faqs_data):
    texto = " ".join([str(v) for v in faq.values()])

    documentos_vectoriales.append({
        "id": f"faq_{i}",
        "texto": texto,
        "metadata": {
            "fuente": "faqs",
            "tipo": "pregunta_frecuente"
        }
    })

# 2) Manuales de productos
# Los manuales sirven para consultas sobre uso, mantenimiento y especificaciones.
for archivo in manuales_path.glob("*.md"):
    with open(archivo, "r", encoding="utf-8") as f:
        texto = f.read()

    documentos_vectoriales.append({
        "id": f"manual_{archivo.stem}",
        "texto": texto,
        "metadata": {
            "fuente": "manuales_productos",
            "tipo": "manual",
            "archivo": archivo.name
        }
    })

# 3) Reseñas de usuarios
# Las reseñas sirven para preguntas sobre opiniones y experiencia de usuarios.
for archivo in resenas_path.glob("*.txt"):
    with open(archivo, "r", encoding="utf-8") as f:
        texto = f.read()

    documentos_vectoriales.append({
        "id": f"resena_{archivo.stem}",
        "texto": texto,
        "metadata": {
            "fuente": "resenas_usuarios",
            "tipo": "resena",
            "archivo": archivo.name
        }
    })

# 4) Tickets de soporte
# Usamos la descripción textual de los tickets para recuperar problemas similares.
for _, row in tickets_df.iterrows():
    texto = f"""
    Producto: {row['nombre_producto']}
    Tipo de problema: {row['tipo_problema']}
    Descripción: {row['descripcion']}
    Severidad: {row['severidad']}
    Categoría: {row['categoria']}
    Estado: {row['estado']}
    Garantía válida: {row['garantia_valida']}
    """

    documentos_vectoriales.append({
        "id": f"ticket_{row['id_ticket']}",
        "texto": texto,
        "metadata": {
            "fuente": "tickets_soporte",
            "tipo": "ticket",
            "id_producto": row["id_producto"],
            "nombre_producto": row["nombre_producto"],
            "categoria": row["categoria"]
        }
    })
print("Documentos preparados para ChromaDB:", len(documentos_vectoriales))


Documentos preparados para ChromaDB: 10065


## Base de datos vectorial con ChromaDB

In [10]:
# Modelo multilingüe apto para español.
# Es chico, rápido y sirve para búsqueda semántica.
embedding_model_name = "intfloat/multilingual-e5-small"

embedding_model = SentenceTransformer(embedding_model_name)

print("Modelo cargado:", embedding_model_name)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Modelo cargado: intfloat/multilingual-e5-small


###  Segmentación de documentos

In [11]:
# Dividimos los textos largos en fragmentos más chicos.
# Esto ayuda a que ChromaDB recupere partes más puntuales y no documentos enormes.

text_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=700,
    chunk_overlap=100
)

fragmentos_vectoriales = []

for doc in documentos_vectoriales:
    partes = text_splitter.split_text(doc["texto"])

    for i, parte in enumerate(partes):
        fragmentos_vectoriales.append({
            "id": f"{doc['id']}_chunk_{i}",
            "texto": parte,
            "metadata": {
                **doc["metadata"],
                "doc_id": doc["id"],
                "chunk": i
            }
        })

print("Documentos originales:", len(documentos_vectoriales))
print("Fragmentos generados:", len(fragmentos_vectoriales))

Documentos originales: 10065
Fragmentos generados: 10575


### Creación de la colección en ChromaDB

In [12]:
# Creamos el cliente local de ChromaDB.
# El cliente es el objeto que administra las colecciones.
client = chromadb.Client()

# Nombre de la colección donde vamos a guardar los fragmentos.
collection_name = "electrodomesticos_docs"

# Si la colección ya existía de una ejecución anterior, la borra.
# Esto evita cargar documentos duplicados si volvemos a correr la celda.
try:
    client.delete_collection(name=collection_name)
except:
    pass

# Creamos la colección nueva.
collection = client.create_collection(name=collection_name)

print("Colección creada:", collection_name)

Colección creada: electrodomesticos_docs


### Generación de embeddings

In [13]:
# Separamos la información en listas porque ChromaDB trabaja con listas paralelas:
# ids: identificadores únicos
# documents: textos
# metadatas: información extra de cada fragmento

ids = [frag["id"] for frag in fragmentos_vectoriales]
documents = [frag["texto"] for frag in fragmentos_vectoriales]
metadatas = [frag["metadata"] for frag in fragmentos_vectoriales]

# Como usamos el modelo E5, agregamos "passage:" delante de los documentos.
# Esto ayuda al modelo a entender que estos textos son pasajes a recuperar.
documents_for_embedding = ["passage: " + texto for texto in documents]

# Generamos los embeddings.
# batch_size indica cuántos textos procesa juntos.
# normalize_embeddings=True deja los vectores normalizados para comparación semántica.
embeddings = embedding_model.encode(
    documents_for_embedding,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
).tolist()

print("Embeddings generados:", len(embeddings))

Batches:   0%|          | 0/166 [00:00<?, ?it/s]

Embeddings generados: 10575


### Carga de fragmentos en ChromaDB

In [14]:
# Cargo los fragmentos en ChromaDB.
# Lo hago en tandas para no mandar todo junto y evitar problemas de memoria.

batch_size = 1000

for i in range(0, len(documents), batch_size):

    collection.add(
        ids=ids[i:i+batch_size],
        documents=documents[i:i+batch_size],
        metadatas=metadatas[i:i+batch_size],
        embeddings=embeddings[i:i+batch_size]
    )

    print(f"Cargados {min(i + batch_size, len(documents))} de {len(documents)}")

print("Carga finalizada.")

Cargados 1000 de 10575
Cargados 2000 de 10575
Cargados 3000 de 10575
Cargados 4000 de 10575
Cargados 5000 de 10575
Cargados 6000 de 10575
Cargados 7000 de 10575
Cargados 8000 de 10575
Cargados 9000 de 10575
Cargados 10000 de 10575
Cargados 10575 de 10575
Carga finalizada.


In [15]:
# Verificación de carga en ChromaDB
print("Cantidad de fragmentos en ChromaDB:", collection.count())

Cantidad de fragmentos en ChromaDB: 10575


### Interfaz de búsqueda en ChromaDB

In [16]:
def buscar_en_chroma(consulta, k=5, filtros=None):
    # Como uso el modelo E5, la consulta lleva el prefijo "query:"
    # Esto le indica al modelo que este texto es una pregunta del usuario.
    consulta_embedding = embedding_model.encode(
        ["query: " + consulta],
        normalize_embeddings=True
    ).tolist()

    # Busco en la colección de ChromaDB.
    # n_results=k indica cuántos fragmentos quiero recuperar.
    # where=filtros permite limitar la búsqueda por metadata.
    resultados = collection.query(
        query_embeddings=consulta_embedding,
        n_results=k,
        where=filtros
    )

    return resultados

### Prueba de búsqueda semántica

In [17]:
consulta = "¿Cómo uso mi licuadora para hacer smoothies?"

resultados = buscar_en_chroma(consulta, k=5)

for i in range(len(resultados["documents"][0])):
    print(f"\n--- Resultado {i+1} ---")
    print("ID:", resultados["ids"][0][i])
    print("Metadata:", resultados["metadatas"][0][i])
    print("Distancia:", resultados["distances"][0][i])
    print(resultados["documents"][0][i][:700])


--- Resultado 1 ---
ID: faq_14_chunk_0
Metadata: {'chunk': 0, 'fuente': 'faqs', 'doc_id': 'faq_14', 'tipo': 'pregunta_frecuente'}
Distancia: 0.2770891785621643
FAQ00015 P0002 Licuadora Uso ¿Puedo usarlo todos los días? El Licuadora de TechHome está diseñado para uso doméstico. Revise el manual del producto (código P0002) para más detalles. Ante cualquier duda, contacte a nuestro servicio de atención al cliente. 2025-02-27 994 77

--- Resultado 2 ---
ID: faq_6_chunk_0
Metadata: {'chunk': 0, 'tipo': 'pregunta_frecuente', 'fuente': 'faqs', 'doc_id': 'faq_6'}
Distancia: 0.2807939946651459
FAQ00007 P0001 Licuadora Uso ¿Cómo se usa correctamente este producto? El Licuadora de TechHome está diseñado para uso doméstico. Revise el manual del producto (código P0001) para más detalles. Ante cualquier duda, contacte a nuestro servicio de atención al cliente. 2025-05-19 3324 66

--- Resultado 3 ---
ID: faq_3_chunk_0
Metadata: {'doc_id': 'faq_3', 'chunk': 0, 'tipo': 'pregunta_frecuente', 'fuente': 

In [20]:
### Prueba con filtro por tipo de documento
consulta = "¿Qué opinan los usuarios de esta cafetera?"

resultados = buscar_en_chroma(
    consulta=consulta,
    k=5,
    filtros={"tipo": "resena"}
)

for i in range(len(resultados["documents"][0])):
    print(f"\n- Resultado {i+1} - ")
    print("ID:", resultados["ids"][0][i])
    print("Metadata:", resultados["metadatas"][0][i])
    print("Distancia:", resultados["distances"][0][i])
    print(resultados["documents"][0][i][:700])


- Resultado 1 - 
ID: resena_resena_R04450_chunk_0
Metadata: {'archivo': 'resena_R04450.txt', 'chunk': 0, 'tipo': 'resena', 'fuente': 'resenas_usuarios', 'doc_id': 'resena_resena_R04450'}
Distancia: 0.26765725016593933
Fecha: 2025-07-30
Usuario: Renata_Reyes
Teléfono: +54 9 37 8802-7521
Producto: Cafetera (P0122)
Puntaje: 3/5
Provincia: Santa Fe
Les cuento... Tiene sus pros y contras con Cafetera. Tiene cosas buenas y cosas malas. Por el precio está bien, pero no esperen maravillas. Espero les sirva!

- Resultado 2 - 
ID: resena_resena_R00776_chunk_0
Metadata: {'fuente': 'resenas_usuarios', 'chunk': 0, 'archivo': 'resena_R00776.txt', 'tipo': 'resena', 'doc_id': 'resena_resena_R00776'}
Distancia: 0.27358895540237427
Fecha: 2024-06-09
Usuario: Francisco_Martínez
Teléfono: +54 9 169 5741-7234
Producto: Cafetera (P0122)
Puntaje: 4/5
Provincia: Río Negro
Les cuento... Increíble relación calidad-precio con Cafetera. Lo compré hace dos meses y es muy duradero, además de elegante. Mi familia e

### Resumen de la base tabular

In [21]:
# Se arma un resumen de la tabla de productos.
# La idea no es pasarle todos los productos al modelo, sino solamente información útil:
# columnas disponibles, valores posibles y rangos numéricos.

categorias = sorted(productos_df["categoria"].dropna().unique().tolist())
subcategorias = sorted(productos_df["subcategoria"].dropna().unique().tolist())
marcas = sorted(productos_df["marca"].dropna().unique().tolist())
colores = sorted(productos_df["color"].dropna().unique().tolist())
voltajes = sorted(productos_df["voltaje"].dropna().unique().tolist())

precio_min = productos_df["precio_usd"].min()
precio_max = productos_df["precio_usd"].max()

stock_min = productos_df["stock"].min()
stock_max = productos_df["stock"].max()

potencia_min = productos_df["potencia_w"].min()
potencia_max = productos_df["potencia_w"].max()

garantia_min = productos_df["garantia_meses"].min()
garantia_max = productos_df["garantia_meses"].max()

resumen_tabular = f"""
Tabla principal: productos_df

Columnas disponibles:
- id_producto
- nombre
- categoria
- subcategoria
- marca
- precio_usd
- stock
- color
- potencia_w
- capacidad
- voltaje
- peso_kg
- garantia_meses
- descripcion

Valores posibles:
- categoria: {categorias}
- subcategoria: {subcategorias}
- marca: {marcas}
- color: {colores}
- voltaje: {voltajes}

Rangos numéricos:
- precio_usd: mínimo {precio_min}, máximo {precio_max}
- stock: mínimo {stock_min}, máximo {stock_max}
- potencia_w: mínimo {potencia_min}, máximo {potencia_max}
- garantia_meses: mínimo {garantia_min}, máximo {garantia_max}
"""

print(resumen_tabular)


Tabla principal: productos_df

Columnas disponibles:
- id_producto
- nombre
- categoria
- subcategoria
- marca
- precio_usd
- stock
- color
- potencia_w
- capacidad
- voltaje
- peso_kg
- garantia_meses
- descripcion

Valores posibles:
- categoria: ['Audio y Video', 'Climatización', 'Cocina', 'Lavado']
- subcategoria: ['Aires Acondicionados', 'Calefacción', 'Cocción', 'Lavado de Ropa', 'Lavado de Vajilla', 'Pequeños Electrodomésticos', 'Planchado', 'Preparación', 'Purificación', 'Refrigeración', 'Secado', 'Televisores', 'Ventilación']
- marca: ['AirFlow', 'ChefMaster', 'CleanMaster', 'ClimaTech', 'CookElite', 'EcoClima', 'FreshWash', 'HomeChef', 'KitchenPro', 'LaundryTech', 'PureAir', 'ScreenPro', 'SparkleHome', 'TechHome', 'ThermoControl', 'VisionPro', 'WashPro']
- color: ['Amarillo', 'Azul', 'Blanco', 'Dorado', 'Gris', 'Negro', 'Plateado', 'Rojo', 'Rosa', 'Verde']
- voltaje: ['110-220V', '12V', '220V']

Rangos numéricos:
- precio_usd: mínimo 28.22, máximo 2992.33
- stock: mínimo 1, m

### Interfaz de búsqueda tabular

In [26]:
def buscar_productos_tabular(filtros, top_k=10):
    # Se copia el DataFrame para no modificar la tabla original.
    df = productos_df.copy()

    # Filtro por texto contenido en el nombre del producto.
    if filtros.get("nombre_contiene"):
        texto = filtros["nombre_contiene"].lower()
        df = df[df["nombre"].str.lower().str.contains(texto, na=False)]

    # Filtro por categoría exacta.
    if filtros.get("categoria"):
        categoria = filtros["categoria"].lower()
        df = df[df["categoria"].str.lower() == categoria]

    # Filtro por subcategoría exacta.
    if filtros.get("subcategoria"):
        subcategoria = filtros["subcategoria"].lower()
        df = df[df["subcategoria"].str.lower() == subcategoria]

    # Filtro por marca exacta.
    if filtros.get("marca"):
        marca = filtros["marca"].lower()
        df = df[df["marca"].str.lower() == marca]

    # Filtro por precio máximo.
    if filtros.get("precio_max") is not None:
        df = df[df["precio_usd"] <= filtros["precio_max"]]

    # Filtro por precio mínimo.
    if filtros.get("precio_min") is not None:
        df = df[df["precio_usd"] >= filtros["precio_min"]]

    # Filtro por stock mínimo.
    if filtros.get("stock_min") is not None:
        df = df[df["stock"] >= filtros["stock_min"]]

    # Filtro por voltaje exacto.
    if filtros.get("voltaje"):
        voltaje = filtros["voltaje"].lower()
        df = df[df["voltaje"].str.lower() == voltaje]

    # Ordenamiento opcional.
    ordenar_por = filtros.get("ordenar_por")
    ascendente = filtros.get("ascendente", True)

    if ordenar_por in df.columns:
        df = df.sort_values(by=ordenar_por, ascending=ascendente)

    # Columnas que se devuelven como resultado.
    columnas_salida = [
        "id_producto", "nombre", "categoria", "subcategoria",
        "marca", "precio_usd", "stock", "voltaje", "garantia_meses"
    ]

    resultado = df[columnas_salida].head(top_k)

    if resultado.empty:
        print("No se encontraron productos con esos filtros.")

    return resultado

### Prueba de búsqueda tabular

In [29]:
# Ejemplo de filtros ya estructurados.
# Más adelante estos filtros serán generados por el modelo local.

filtros_prueba = {
    "nombre_contiene": "licuadora",
    "precio_max": 400,
    "stock_min": 1,
    "ordenar_por": "precio_usd",
    "ascendente": True
}

resultado_tabular = buscar_productos_tabular(filtros_prueba, top_k=10)

display(resultado_tabular)

,id_producto,nombre,categoria,subcategoria,marca,precio_usd,stock,voltaje,garantia_meses
3,P0004,Compacto Licuadora,Cocina,Preparación,ChefMaster,259.42,75,220V,24
0,P0001,Licuadora,Cocina,Preparación,TechHome,283.63,108,12V,36
2,P0003,Plus Licuadora Pro,Cocina,Preparación,TechHome,329.07,97,220V,18


##  Base de datos de grafos con GrafitoDB

###  Preparación de relaciones desde DataFrame

In [30]:
# Se prepara un DataFrame de relaciones para la base de grafos.
# Cada fila representa una relación entre dos nodos.

relaciones = []

for _, row in productos_df.iterrows():
    id_producto = row["id_producto"]
    nombre_producto = row["nombre"]
    categoria = row["categoria"]
    subcategoria = row["subcategoria"]
    marca = row["marca"]

    # Producto -> Categoría
    relaciones.append({
        "origen_tipo": "Producto",
        "origen_id": id_producto,
        "origen_nombre": nombre_producto,
        "relacion": "PERTENECE_A",
        "destino_tipo": "Categoria",
        "destino_id": categoria,
        "destino_nombre": categoria
    })

    # Producto -> Subcategoría
    relaciones.append({
        "origen_tipo": "Producto",
        "origen_id": id_producto,
        "origen_nombre": nombre_producto,
        "relacion": "TIENE_SUBCATEGORIA",
        "destino_tipo": "Subcategoria",
        "destino_id": subcategoria,
        "destino_nombre": subcategoria
    })

    # Producto -> Marca
    relaciones.append({
        "origen_tipo": "Producto",
        "origen_id": id_producto,
        "origen_nombre": nombre_producto,
        "relacion": "TIENE_MARCA",
        "destino_tipo": "Marca",
        "destino_id": marca,
        "destino_nombre": marca
    })

relaciones_grafo_df = pd.DataFrame(relaciones)

print("Relaciones preparadas:", relaciones_grafo_df.shape)
display(relaciones_grafo_df.head(10))

Relaciones preparadas: (900, 7)


,origen_tipo,origen_id,origen_nombre,relacion,destino_tipo,destino_id,destino_nombre
0,Producto,P0001,Licuadora,PERTENECE_A,Categoria,Cocina,Cocina
1,Producto,P0001,Licuadora,TIENE_SUBCATEGORIA,Subcategoria,Preparación,Preparación
2,Producto,P0001,Licuadora,TIENE_MARCA,Marca,TechHome,TechHome
3,Producto,P0002,Licuadora,PERTENECE_A,Categoria,Cocina,Cocina
4,Producto,P0002,Licuadora,TIENE_SUBCATEGORIA,Subcategoria,Preparación,Preparación
5,Producto,P0002,Licuadora,TIENE_MARCA,Marca,TechHome,TechHome
6,Producto,P0003,Plus Licuadora Pro,PERTENECE_A,Categoria,Cocina,Cocina
7,Producto,P0003,Plus Licuadora Pro,TIENE_SUBCATEGORIA,Subcategoria,Preparación,Preparación
8,Producto,P0003,Plus Licuadora Pro,TIENE_MARCA,Marca,TechHome,TechHome
9,Producto,P0004,Compacto Licuadora,PERTENECE_A,Categoria,Cocina,Cocina


### Creación de la base GrafitoDB

In [34]:
# Se crea una base de grafos en memoria.
# La base queda disponible mientras dure la sesión de Colab.

db_grafo = GrafitoDatabase(':memory:', cypher_max_hops=6)

print("Base GrafitoDB creada.")

Base GrafitoDB creada.


### Carga de nodos y relaciones en GrafitoDB

In [35]:
# Diccionario auxiliar para no crear nodos duplicados.
# La clave será una combinación de tipo de nodo e id.

nodos_grafo = {}

def obtener_o_crear_nodo(tipo, nodo_id, nombre):
    # Se arma una clave única para cada nodo.
    clave = (tipo, str(nodo_id))

    # Si el nodo ya fue creado, se reutiliza.
    if clave in nodos_grafo:
        return nodos_grafo[clave]

    # Si no existe, se crea en GrafitoDB.
    nodo = db_grafo.create_node(
        labels=[tipo],
        properties={
            "id": str(nodo_id),
            "nombre": str(nombre)
        }
    )

    nodos_grafo[clave] = nodo
    return nodo


# Se cargan en GrafitoDB las relaciones preparadas en relaciones_grafo_df.
for _, row in relaciones_grafo_df.iterrows():
    origen = obtener_o_crear_nodo(
        row["origen_tipo"],
        row["origen_id"],
        row["origen_nombre"]
    )

    destino = obtener_o_crear_nodo(
        row["destino_tipo"],
        row["destino_id"],
        row["destino_nombre"]
    )

    db_grafo.create_relationship(
        origen.id,
        destino.id,
        row["relacion"]
    )

print("Nodos creados:", len(nodos_grafo))
print("Relaciones cargadas:", len(relaciones_grafo_df))

Nodos creados: 334
Relaciones cargadas: 900


### Consultas Cypher sobre GrafitoDB

In [36]:
# Consulta de productos asociados a una categoría.

consulta = """
MATCH (p:Producto)-[:PERTENECE_A]->(c:Categoria {nombre: 'Cocina'})
RETURN p.nombre, c.nombre
"""

resultado = db_grafo.execute(consulta)

for fila in resultado[:10]:
    print(f"Producto: {fila['p.nombre']} | Categoría: {fila['c.nombre']}")

Producto: Licuadora | Categoría: Cocina
Producto: Licuadora | Categoría: Cocina
Producto: Plus Licuadora Pro | Categoría: Cocina
Producto: Compacto Licuadora | Categoría: Cocina
Producto: Licuadora | Categoría: Cocina
Producto: Licuadora | Categoría: Cocina
Producto: Licuadora | Categoría: Cocina
Producto: Ultra Licuadora | Categoría: Cocina
Producto: Procesadora | Categoría: Cocina
Producto: Deluxe Procesadora | Categoría: Cocina


In [37]:
# Consulta de productos asociados a una marca.

consulta = """
MATCH (p:Producto)-[:TIENE_MARCA]->(m:Marca {nombre: 'TechHome'})
RETURN p.nombre, m.nombre
"""

resultado = db_grafo.execute(consulta)

for fila in resultado[:10]:
    print(f"Producto: {fila['p.nombre']} | Marca: {fila['m.nombre']}")

Producto: Licuadora | Marca: TechHome
Producto: Licuadora | Marca: TechHome
Producto: Plus Licuadora Pro | Marca: TechHome
Producto: Ultra Licuadora | Marca: TechHome
Producto: Deluxe Procesadora | Marca: TechHome
Producto: Advanced Procesadora II | Marca: TechHome
Producto: Profesional Batidora de Mano | Marca: TechHome
Producto: Profesional Mixer | Marca: TechHome
Producto: Digital Rallador Eléctrico | Marca: TechHome
Producto: Eco Rallador Eléctrico | Marca: TechHome


In [38]:
# Consulta de productos asociados a una subcategoría.

consulta = """
MATCH (p:Producto)-[:TIENE_SUBCATEGORIA]->(s:Subcategoria {nombre: 'Preparación'})
RETURN p.nombre, s.nombre
"""

resultado = db_grafo.execute(consulta)

for fila in resultado[:10]:
    print(f"Producto: {fila['p.nombre']} | Subcategoría: {fila['s.nombre']}")

Producto: Licuadora | Subcategoría: Preparación
Producto: Licuadora | Subcategoría: Preparación
Producto: Plus Licuadora Pro | Subcategoría: Preparación
Producto: Compacto Licuadora | Subcategoría: Preparación
Producto: Licuadora | Subcategoría: Preparación
Producto: Licuadora | Subcategoría: Preparación
Producto: Licuadora | Subcategoría: Preparación
Producto: Ultra Licuadora | Subcategoría: Preparación
Producto: Procesadora | Subcategoría: Preparación
Producto: Deluxe Procesadora | Subcategoría: Preparación


### Interfaz de consulta para GrafitoDB

In [39]:
def buscar_en_grafo(query_cypher, limite=10):
    # Ejecuta una consulta Cypher sobre GrafitoDB.
    # Devuelve como máximo la cantidad de resultados indicada en limite.

    resultado = db_grafo.execute(query_cypher)

    return resultado[:limite]

In [40]:
query_grafo_prueba = """
MATCH (p:Producto)-[:PERTENECE_A]->(c:Categoria {nombre: 'Cocina'})
RETURN p.nombre, c.nombre
"""

resultado_grafo = buscar_en_grafo(query_grafo_prueba, limite=5)

for fila in resultado_grafo:
    print(f"Producto: {fila['p.nombre']} | Categoría: {fila['c.nombre']}")

Producto: Licuadora | Categoría: Cocina
Producto: Licuadora | Categoría: Cocina
Producto: Plus Licuadora Pro | Categoría: Cocina
Producto: Compacto Licuadora | Categoría: Cocina
Producto: Licuadora | Categoría: Cocina
